# 02 — MultiROCKET

MultiROCKET feature extraction and persistence. Uses `src.features.multirocket` and `src.features.memmap`.

In [1]:
# Path fix: ensure we use this repo's src (not e.g. OneDrive)
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

import logging
from src.utils import SEED, set_global_seed
from src.utils.paths import get_multirocket_features_dir, get_models_dir, get_splits_dir, ensure_dir
from src.data import load_raw_dataset, create_splits, load_splits, save_splits
from src.features.multirocket import (
    create_multirocket_transformer,
    fit_multirocket,
    transform_multirocket_batched,
    save_multirocket_transformer,
)
from src.features.memmap import write_array_to_memmap, open_memmap_read

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
print("Using PROJECT_ROOT:", PROJECT_ROOT)
print("SEED:", SEED)

Using PROJECT_ROOT: C:\Projects\DeepLearningProject
SEED: 0


In [2]:
# Constants: 60/20/20 splits, MultiROCKET params
TEST_SIZE = 0.2
VAL_SIZE = 0.2
TASK_TYPE = "multilabel"
NUM_KERNELS = 2048
BATCH_SIZE = 1024
print("Split: test_size=%s val_size=%s (60/20/20) | num_kernels=%s batch_size=%s" % (TEST_SIZE, VAL_SIZE, NUM_KERNELS, BATCH_SIZE))

Split: test_size=0.2 val_size=0.2 (60/20/20) | num_kernels=2048 batch_size=1024


In [3]:
# Load data and set seed
print("Loading dataset from cache...")
X, y = load_raw_dataset()
print("Loaded X shape %s, y shape %s" % (X.shape, y.shape))

set_global_seed(SEED)
print("Global seed set to %s" % SEED)

Loading dataset from cache...


2026-02-27 23:14:26,158 | INFO | Loaded cached dataset C:\Projects\DeepLearningProject\data\processed\chapman_wfdb_Xy.npz | X=(45150, 12, 5000) y=(45150, 4) | 23.35s


Loaded X shape (45150, 12, 5000), y shape (45150, 4)


2026-02-27 23:14:29,165 | INFO | Global seed set to 0


Global seed set to 0


In [4]:
# Splits: load existing or create 60/20/20
splits_path = get_splits_dir() / "splits.npz"
if splits_path.exists():
    print("Loading splits (60/20/20) from %s..." % splits_path)
    idx_train, idx_val, idx_test = load_splits(splits_path)
else:
    print("Creating splits (60/20/20)...")
    idx_train, idx_val, idx_test = create_splits(y, task_type=TASK_TYPE, test_size=TEST_SIZE, val_size=VAL_SIZE, seed=SEED)
    save_splits(idx_train, idx_val, idx_test, splits_path)
print("Train %d, val %d, test %d" % (len(idx_train), len(idx_val), len(idx_test)))

Loading splits (60/20/20) from C:\Projects\DeepLearningProject\data\processed\splits\splits.npz...
Train 36120, val 2257, test 6773


In [5]:
# Create and fit MultiROCKET on training data only
print("Fitting MultiROCKET on %d samples..." % len(idx_train))
transformer = create_multirocket_transformer(num_kernels=NUM_KERNELS, seed=SEED)
X_train = X[idx_train]
fit_multirocket(transformer, X_train)
print("Fit complete.")

Fitting MultiROCKET on 36120 samples...


2026-02-27 23:16:31,492 | INFO | Fitted MultiROCKET on train shape (36120, 12, 5000)


Fit complete.


In [6]:
# Transform train/val/test in batches (progress logged per batch)
print("Transforming train set...")
F_train = transform_multirocket_batched(transformer, X[idx_train], batch_size=BATCH_SIZE)
print("Transforming val set...")
F_val = transform_multirocket_batched(transformer, X[idx_val], batch_size=BATCH_SIZE) if len(idx_val) > 0 else None
print("Transforming test set...")
F_test = transform_multirocket_batched(transformer, X[idx_test], batch_size=BATCH_SIZE)
print("F_train %s, F_val %s, F_test %s" % (F_train.shape, F_val.shape if F_val is not None else None, F_test.shape))

Transforming train set...


2026-02-27 23:19:04,561 | INFO | Transform batch 1/36 (samples 1024/36120) | 83.7s elapsed | ETA 2868s
2026-02-27 23:20:25,100 | INFO | Transform batch 2/36 (samples 2048/36120) | 164.2s elapsed | ETA 2732s
2026-02-27 23:21:46,805 | INFO | Transform batch 3/36 (samples 3072/36120) | 245.9s elapsed | ETA 2646s
2026-02-27 23:23:09,741 | INFO | Transform batch 4/36 (samples 4096/36120) | 328.9s elapsed | ETA 2571s
2026-02-27 23:24:28,902 | INFO | Transform batch 5/36 (samples 5120/36120) | 408.0s elapsed | ETA 2470s
2026-02-27 23:25:47,957 | INFO | Transform batch 6/36 (samples 6144/36120) | 487.1s elapsed | ETA 2376s
2026-02-27 23:27:07,276 | INFO | Transform batch 7/36 (samples 7168/36120) | 566.4s elapsed | ETA 2288s
2026-02-27 23:28:26,719 | INFO | Transform batch 8/36 (samples 8192/36120) | 645.8s elapsed | ETA 2202s
2026-02-27 23:29:46,352 | INFO | Transform batch 9/36 (samples 9216/36120) | 725.5s elapsed | ETA 2118s
2026-02-27 23:31:05,275 | INFO | Transform batch 10/36 (samples 1

Transforming val set...


2026-02-28 00:05:45,075 | INFO | Transform batch 1/3 (samples 1024/2257) | 79.0s elapsed | ETA 95s
2026-02-28 00:07:04,008 | INFO | Transform batch 2/3 (samples 2048/2257) | 158.0s elapsed | ETA 16s
2026-02-28 00:07:19,985 | INFO | Transform batch 3/3 (samples 2257/2257) | 174.0s elapsed | ETA 0s
2026-02-28 00:07:20,008 | INFO | Transformed shape (2257, 12, 5000) -> (2257, 16128) in 174.0s


Transforming test set...


2026-02-28 00:08:47,801 | INFO | Transform batch 1/7 (samples 1024/6773) | 79.6s elapsed | ETA 447s
2026-02-28 00:10:05,066 | INFO | Transform batch 2/7 (samples 2048/6773) | 156.9s elapsed | ETA 362s
2026-02-28 00:11:22,230 | INFO | Transform batch 3/7 (samples 3072/6773) | 234.0s elapsed | ETA 282s
2026-02-28 00:12:39,791 | INFO | Transform batch 4/7 (samples 4096/6773) | 311.6s elapsed | ETA 204s
2026-02-28 00:13:57,683 | INFO | Transform batch 5/7 (samples 5120/6773) | 389.5s elapsed | ETA 126s
2026-02-28 00:15:15,413 | INFO | Transform batch 6/7 (samples 6144/6773) | 467.2s elapsed | ETA 48s
2026-02-28 00:16:02,558 | INFO | Transform batch 7/7 (samples 6773/6773) | 514.4s elapsed | ETA 0s
2026-02-28 00:16:02,637 | INFO | Transformed shape (6773, 12, 5000) -> (6773, 16128) in 514.4s


F_train (36120, 16128), F_val (2257, 16128), F_test (6773, 16128)


In [7]:
# Save transformer
mr_transformer_path = get_models_dir() / "multirocket" / "transformer.joblib"
ensure_dir(mr_transformer_path.parent)
print("Saving transformer to %s..." % mr_transformer_path)
save_multirocket_transformer(transformer, mr_transformer_path)
print("Saved.")

2026-02-28 00:16:02,821 | INFO | Saved MultiROCKET transformer to C:\Projects\DeepLearningProject\models\multirocket\transformer.joblib


Saving transformer to C:\Projects\DeepLearningProject\models\multirocket\transformer.joblib...
Saved.


In [8]:
# Save features (memmap) and shape metadata
mr_dir = ensure_dir(get_multirocket_features_dir())
print("Writing train.dat (shape %s)..." % (F_train.shape,))
write_array_to_memmap(mr_dir / "train.dat", F_train)
print("Writing test.dat (shape %s)..." % (F_test.shape,))
write_array_to_memmap(mr_dir / "test.dat", F_test)
if F_val is not None:
    print("Writing val.dat (shape %s)..." % (F_val.shape,))
    write_array_to_memmap(mr_dir / "val.dat", F_val)

# Persist shapes for reopening memmaps
import numpy as np
shapes = {"train": np.array(F_train.shape), "test": np.array(F_test.shape)}
if F_val is not None:
    shapes["val"] = np.array(F_val.shape)
np.savez(mr_dir / "shapes.npz", **shapes)
print("Saved shapes to %s" % (mr_dir / "shapes.npz"))
print("Done. Features in %s" % mr_dir)

Writing train.dat (shape (36120, 16128))...


2026-02-28 00:16:04,657 | INFO | Wrote array shape (36120, 16128) to memmap C:\Projects\DeepLearningProject\data\processed\multirocket\train.dat


Writing test.dat (shape (6773, 16128))...


2026-02-28 00:16:04,955 | INFO | Wrote array shape (6773, 16128) to memmap C:\Projects\DeepLearningProject\data\processed\multirocket\test.dat
2026-02-28 00:16:05,064 | INFO | Wrote array shape (2257, 16128) to memmap C:\Projects\DeepLearningProject\data\processed\multirocket\val.dat


Writing val.dat (shape (2257, 16128))...
Saved shapes to C:\Projects\DeepLearningProject\data\processed\multirocket\shapes.npz
Done. Features in C:\Projects\DeepLearningProject\data\processed\multirocket


## Inspect MultiROCKET features (variance & constant columns)

In [4]:
import numpy as np

# Use in-memory F_train if already computed above; otherwise load from memmap
try:
    F_train
except NameError:
    from src.utils.paths import get_multirocket_features_dir
    from src.features.memmap import open_memmap_read
    mr_dir = get_multirocket_features_dir()
    shapes_path = mr_dir / "shapes.npz"
    if not shapes_path.exists():
        raise FileNotFoundError(
            "MultiROCKET features not in memory (F_train) and not on disk at %s. "
            "Run the transform and save cells above in order, or run this notebook from the project "
            "that contains data/processed/multirocket/ (e.g. the repo where you ran 02_multirocket)." % mr_dir
        )
    shapes = np.load(shapes_path)
    F_train = open_memmap_read(mr_dir / "train.dat", tuple(shapes["train"]))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\jacob\\OneDrive\\Documents\\School\\Spring 2026\\Deep Learning Project\\data\\processed\\multirocket\\shapes.npz'

In [ ]:
import numpy as np

# Use training set for variance (avoids leakage; train defines "baseline" variability)
F = F_train  # shape (n_samples, n_features)
n_samples, n_features = F.shape

# Per-column stats
col_mean = np.mean(F, axis=0)
col_std = np.std(F, axis=0)
col_var = np.var(F, axis=0)

# Thresholds (tune if needed)
VAR_ZERO = 0.0
VAR_NEAR_ZERO = 1e-10  # or 1e-12 for float32

zero_var = np.where(col_var <= VAR_ZERO)[0]
near_zero_var = np.where((col_var > VAR_ZERO) & (col_var <= VAR_NEAR_ZERO))[0]

print("MultiROCKET feature inspection (train set)")
print("  Shape: %s" % (F.shape,))
print("  Zero variance (var=0):     %d columns" % len(zero_var))
print("  Near-zero (var<=%g): %d columns" % (VAR_NEAR_ZERO, len(near_zero_var)))
if len(zero_var) > 0:
    print("  Zero-var column indices (first 50):", zero_var[:50].tolist())
if len(near_zero_var) > 0:
    print("  Near-zero-var column indices (first 50):", near_zero_var[:50].tolist())

# Optional: variance distribution
print("\nVariance percentiles (train):")
for p in [0, 1, 5, 25, 50, 75, 95, 99, 100]:
    print("  p%d = %g" % (p, np.percentile(col_var, p)))